In [ ]:
import os
import zipfile
from google.colab import files

# 1. Upload your 'kaggle.json' file
# If you already uploaded it, this part skips automatically.
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your 'kaggle.json' file now...")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
else:
    print("Kaggle key found. Skipping upload.")

# 2. Download Dataset
if not os.path.exists("/content/dataset"):
    print("Downloading Dataset...")
    !kaggle datasets download -d ashenafifasilkebede/dataset -p /content/dataset
    print("Unzipping...")
    with zipfile.ZipFile("/content/dataset/dataset.zip", 'r') as zip_ref:
        zip_ref.extractall("/content/dataset")
    print("Dataset Downloaded & Unzipped!")
else:
    print("Dataset already exists.")

In [ ]:
import tensorflow as tf
import os

# 1. Find the Correct Folder
# We look for the folder that actually contains 'Normal' and 'OSCC'
DATA_DIR = None
print("Searching for data folder...")
for root, dirs, files in os.walk("/content/dataset"):
    if {'Normal', 'OSCC'}.issubset(set(dirs)):
        DATA_DIR = root
        break

if not DATA_DIR:
    DATA_DIR = "/content/dataset/train" # Fallback
    print("Warning: Using fallback path.")
else:
    print(f"Data found at: {DATA_DIR}")

# 2. Load Data
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# 3. SAVE CLASS NAMES NOW (Crucial Step)
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Classes Saved: {class_names}")

# 4. Optimize (Now it's safe to do this)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# UNIT TESTING: PREPROCESSING

# Get one batch from dataset
for images, labels in train_ds.take(1):
    sample_image = images[0]
    sample_label = labels[0]

# Check image properties
print("Image Shape:", sample_image.shape)
print("Pixel Range (Min, Max):", tf.reduce_min(sample_image).numpy(), tf.reduce_max(sample_image).numpy())

# Check label
print("Label (One-hot):", sample_label.numpy())
print("Class Index:", tf.argmax(sample_label).numpy())
print("Class Name:", class_names[tf.argmax(sample_label).numpy()])

In [ ]:
from tensorflow.keras import layers, models, Input
from tensorflow.keras.applications import EfficientNetB0, ResNet50V2

def build_fusion_model(num_classes):
    inputs = Input(shape=(224, 224, 3))

    # DATA AUGMENTATION
    data_augmentation = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ])

    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)

    # EfficientNet Branch
    base_eff = EfficientNetB0(include_top=False, weights="imagenet", input_tensor=x)
    base_eff.trainable = False
    x1 = layers.GlobalAveragePooling2D()(base_eff.output)
    x1 = layers.Dense(128, activation="relu")(x1)

    # ResNet Branch
    base_res = ResNet50V2(include_top=False, weights="imagenet", input_tensor=x)
    base_res.trainable = False
    x2 = layers.GlobalAveragePooling2D()(base_res.output)
    x2 = layers.Dense(128, activation="relu")(x2)

    # Fusion
    merged = layers.Concatenate()([x1, x2])
    c = layers.Dense(128, activation="relu")(merged)
    c = layers.Dropout(0.5)(c)
    outputs = layers.Dense(num_classes, activation="softmax")(c)

    return models.Model(inputs=inputs, outputs=outputs)

# Build and Compile
print(f"Building model for {num_classes} classes...")
fusion_model = build_fusion_model(num_classes)
fusion_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
fusion_model.summary()

In [ ]:
for i, layer in enumerate(fusion_model.layers):
    print(i, layer.name)

In [ ]:
# UNIT TESTING : EfficientNet branch

eff_branch = models.Model(
    inputs=fusion_model.input,
    outputs=fusion_model.layers[-6].output
)

eff_out = eff_branch.predict(sample, verbose=0)

print("EfficientNet Feature Shape:", eff_out.shape)

In [ ]:
# UNIT TESTING : ResNet branch

res_branch = models.Model(
    inputs=fusion_model.input,
    outputs=fusion_model.layers[-5].output
)

res_out = res_branch.predict(sample, verbose=0)

print("ResNet Feature Shape:", res_out.shape)

In [ ]:
# Unit Test: Fusion Layer

fusion_branch = models.Model(
    inputs=fusion_model.input,
    outputs=fusion_model.get_layer("concatenate").output
)

fusion_out = fusion_branch.predict(sample)

print("Fused Feature Shape:", fusion_out.shape)

In [ ]:
# Unit Test: Data Augmentation

import matplotlib.pyplot as plt

augmented = fusion_model.layers[1](sample)

plt.subplot(1,2,1)
plt.imshow(sample[0].numpy().astype("uint8"))
plt.title("Original")

plt.subplot(1,2,2)
plt.imshow(augmented[0].numpy().astype("uint8"))
plt.title("Augmented")

plt.show()

In [ ]:
# Unit Test: Trainable Layers

for layer in fusion_model.layers:
    if layer.trainable:
        print(layer.name)

In [ ]:
from tensorflow.keras import models

feature_extractor = models.Model(
    inputs=fusion_model.input,
    outputs=fusion_model.layers[-2].output
)

In [ ]:
# INTEGRATION TEST 1:
# EfficientNet + ResNet → Fusion

for images, labels in train_ds.take(1):
    sample = images[0:1]

# Pass through fusion model till feature layer
fusion_features = feature_extractor.predict(sample, verbose=0)

print("Fusion Feature Shape:", fusion_features.shape)

In [ ]:
# INTEGRATION TEST 2:
# Fusion → CNN Classifier

import numpy as np

for images, labels in train_ds.take(1):
    sample = images[0:1]

prediction = fusion_model.predict(sample, verbose=0)

print("Prediction:", prediction)
print("Predicted Class:", np.argmax(prediction))

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint("best_fusion_model.keras", save_best_only=True)
]

print("Starting Training...")
history = fusion_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)
print("Training Complete.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score # Added accuracy_score

# 1. Plot Metrics (Graphs)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.plot(history.history['accuracy'], label='Train'); plt.plot(history.history['val_accuracy'], label='Val'); plt.legend(); plt.title("Accuracy")
plt.subplot(1, 2, 2); plt.plot(history.history['loss'], label='Train'); plt.plot(history.history['val_loss'], label='Val'); plt.legend(); plt.title("Loss")
plt.show()

# 2. Confusion Matrix & Accuracy
print("Generating Predictions for Metrics...")
y_true, y_pred = [], []
for img, label in val_ds:
    preds = fusion_model.predict(img, verbose=0)
    y_true.extend(np.argmax(label.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

val_acc = accuracy_score(y_true, y_pred)
print(f"\nFinal Validation Accuracy: {val_acc * 100:.2f}%")
print("-" * 30)

# Plot Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Actual'); plt.xlabel('Predicted'); plt.title('Confusion Matrix')
plt.show()

# 3. VISUALIZATION: 8 PREDICTIONS
print("\nGenerating 8 Prediction Examples...")
plt.figure(figsize=(20, 10))
for images, labels in val_ds.take(1):
    preds = fusion_model.predict(images, verbose=0)
    for i in range(8): # Show first 8
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))

        true_idx = np.argmax(labels[i])
        pred_idx = np.argmax(preds[i])
        conf = 100 * np.max(preds[i])
        col = 'green' if true_idx == pred_idx else 'red'

        plt.title(f"True: {class_names[true_idx]}\nPred: {class_names[pred_idx]} ({conf:.1f}%)", color=col, fontsize=12, fontweight='bold')
        plt.axis("off")
plt.suptitle("Fusion Model Predictions", fontsize=16)
plt.show()

In [ ]:
# Hybrid CNN + XGBoost Complete Stable Version

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import models

feature_extractor = models.Model(
    inputs=fusion_model.input,
    outputs=fusion_model.layers[-2].output
)

def extract_features(ds):
    all_images = []
    all_labels = []

    for img, lbl in ds:
        all_images.append(img)
        all_labels.append(np.argmax(lbl.numpy(), axis=1))

    all_images = tf.concat(all_images, axis=0)
    all_labels = np.concatenate(all_labels)

    features = feature_extractor.predict(all_images, verbose=0)

    return features, all_labels

X_train, y_train = extract_features(train_ds)
X_val, y_val = extract_features(val_ds)

classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))
sample_weights = np.array([class_weight_dict[label] for label in y_train])

xgb_clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.5,
    reg_alpha=0.5,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_clf.fit(X_train, y_train, sample_weight=sample_weights)

y_pred = xgb_clf.predict(X_val)

acc = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, average='weighted')
recall = recall_score(y_val, y_pred, average='weighted')
f1 = f1_score(y_val, y_pred, average='weighted')

print(f"Accuracy  : {acc*100:.2f}%")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=class_names))

cm = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Hybrid CNN + XGBoost")
plt.show()

plt.figure(figsize=(8, 6))
xgb.plot_importance(xgb_clf, max_num_features=15)
plt.title("Top 15 Important Features")
plt.show()

plt.figure(figsize=(20, 10))

for images, labels in val_ds.take(1):
    batch_feats = feature_extractor.predict(images, verbose=0)
    xgb_preds = xgb_clf.predict(batch_feats)

    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))

        true_txt = class_names[np.argmax(labels[i])]
        pred_txt = class_names[xgb_preds[i]]

        color = 'blue' if true_txt == pred_txt else 'red'

        plt.title(f"True: {true_txt}\nXGBoost: {pred_txt}",
                  color=color,
                  fontsize=12,
                  fontweight='bold')

        plt.axis("off")

plt.suptitle("Hybrid CNN + XGBoost Predictions", fontsize=16)
plt.show()


In [ ]:
# =========================
# INTEGRATION TEST 3:
# Fusion Features → XGBoost
# =========================

for images, labels in val_ds.take(1):
    sample = images[0:1]

# Extract deep features
features = feature_extractor.predict(sample, verbose=0)

# XGBoost prediction
xgb_pred = xgb_clf.predict(features)

print("Extracted Feature Shape:", features.shape)
print("XGBoost Prediction:", xgb_pred)
print("Class Name:", class_names[xgb_pred[0]])

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

def get_gradcam(model, img_array):
    # 1. Find the target layer. This is critical.
    # The layer name must exist in the model you are passing.
    target_layer_name = None
    for layer in reversed(model.layers):
        # Look for a common convolutional layer name. Adjust if needed.
        if "top_activation" in layer.name or "post_relu" in layer.name:
            target_layer_name = layer.name
            break

    if target_layer_name is None:
        print("Warning: Could not find a suitable target layer (e.g., 'top_activation').")
        return None

    # 2. Create the gradient model
    try:
        # If the model is a Sequential model wrapping a base model, you might need
        # to access the base model first: base_model = model.get_layer("your_base_model_name")
        # For now, we'll assume 'model' is the one containing the target layer.
        grad_model = tf.keras.models.Model(
            [model.inputs], [model.get_layer(target_layer_name).output, model.output]
        )
    except Exception as e:
        print(f"Error creating grad_model: {e}")
        return None

    # 3. Record operations for gradient calculation
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        # We are interested in class 1 (Cancer)
        loss = preds[:, 1]

    # 4. Calculate gradients
    grads = tape.gradient(loss, conv_out)
    # Pool the gradients across the channels
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # 5. Generate the heatmap
    # Multiply each channel by its importance (pooled gradient)
    heatmap = conv_out[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # --- THE FIX IS HERE ---
    # Apply ReLU to keep only positive contributions
    numerator = tf.maximum(heatmap, 0)
    # Get the maximum value for normalization
    max_val = tf.math.reduce_max(numerator)
    # Add a tiny epsilon to the divisor to prevent division by zero
    denominator = max_val + tf.keras.backend.epsilon()
    # Normalize safely. The result is now guaranteed to be free of NaNs.
    heatmap = numerator / denominator
    # -----------------------

    return heatmap.numpy()

# --- FIND 8 CANCER IMAGES ---
print("Searching for 8 Cancer samples...")
found_imgs = []
count = 0

# NOTE: This requires 'val_ds' to be defined from your previous code
for imgs, lbls in val_ds:
    for k in range(len(lbls)):
        # Assuming labels are one-hot encoded. If they are integers, use `if lbls[k] == 1:`
        if np.argmax(lbls[k]) == 1: # 1 = Cancer
            found_imgs.append(imgs[k].numpy())
            count += 1
        if count >= 8: break
    if count >= 8: break

# --- PLOT ---
if len(found_imgs) > 0:
    plt.figure(figsize=(20, 10))
    for i, sample in enumerate(found_imgs):
        # IMPORTANT: Pass the correct model here. If 'fusion_model' is an ensemble
        # of models, you cannot pass it directly. You must pass one of the
        # underlying base models (e.g., model_eff, model_res) that has convolutional layers.
        # Example: hm = get_gradcam(model_res, np.expand_dims(sample, axis=0))
        # For this correction, I will assume 'fusion_model' is a valid Keras model.
        hm = get_gradcam(fusion_model, np.expand_dims(sample, axis=0))

        if hm is not None:
            # 1. Resize heatmap to match image size
            hm = cv2.resize(hm, (224, 224))

            # 2. Convert to RGB heatmap
            # The fix in get_gradcam ensures this cast is safe
            hm_uint8 = np.uint8(255 * hm)
            heatmap_img = cv2.applyColorMap(hm_uint8, cv2.COLORMAP_JET)

            # 3. Prepare sample image for blending
            # CRITICAL: Check if sample is 0-1 (float) or 0-255 (uint8)
            if np.max(sample) <= 1.0:
                sample_scaled = sample * 255 # Scale to 0-255
            else:
                sample_scaled = sample # Already 0-255

            # Convert to float32 for accurate blending
            sample_float = sample_scaled.astype(np.float32)
            heatmap_float = heatmap_img.astype(np.float32)

            # 4. Blend images
            alpha = 0.4 # Strength of the heatmap
            overlay = heatmap_float * alpha + sample_float
            # Clip back to 0-255 range and convert to uint8
            overlay = np.clip(overlay, 0, 255).astype("uint8")

            plt.subplot(2, 4, i + 1)
            plt.imshow(overlay)
            plt.title(f"Cancer Sample {i+1}", fontsize=12)
            plt.axis("off")
        else:
             print(f"Could not generate Grad-CAM for sample {i+1}. Checking next...")
             # Display original image as a fallback
             plt.subplot(2, 4, i + 1)
             plt.imshow(sample.astype("uint8"))
             plt.title(f"Sample {i+1} (No Heatmap)", fontsize=12)
             plt.axis("off")

    plt.suptitle("Grad-CAM Heatmaps (Red = High Model Focus)", fontsize=16)
    plt.show()
else:
    print("No cancer samples found in the validation batch scanned.")

In [ ]:
from google.colab import files

# 1. Save the model locally in Colab first
fusion_model.save("best_fusion_model.keras")

# 2. Download it to your computer
print("Downloading model to your computer...")
files.download("best_fusion_model.keras")

In [ ]:
# TESTING CODE

import os
import numpy as np
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt

from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# 1. LOAD BEST MODEL
fusion_model = load_model("best_fusion_model.keras")
print("Loaded best model")

# 2. FIND TEST DIRECTORY
TEST_DIR = None
for root, dirs, files in os.walk("/content/dataset"):
    if 'test' in root.lower():
        TEST_DIR = root
        break

if TEST_DIR:
    print(f"Test data found at: {TEST_DIR}")
else:
    raise Exception("Test folder not found")

# 3. LOAD TEST DATASET
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical',
    shuffle=False
)

print("Test dataset loaded")

# 4. CHECK CLASS ORDER
print("Train classes:", class_names)
print("Test classes :", test_ds.class_names)

# 5. TEST WITH TTA + THRESHOLD

threshold = 0.6

y_true = []
y_pred = []

for images, labels in test_ds:

    # TTA predictions
    preds1 = fusion_model.predict(images, verbose=0)
    flipped_images = tf.image.flip_left_right(images)
    preds2 = fusion_model.predict(flipped_images, verbose=0)

    preds = (preds1 + preds2) / 2

    # TRUE labels
    y_true.extend(np.argmax(labels.numpy(), axis=1))

    # THRESHOLD-BASED predictions
    for p in preds:
        if p[1] > threshold:
            y_pred.append(1)
        else:
            y_pred.append(0)

# 6. METRICS

acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print("\nFINAL TEST RESULTS ")
print(f"Accuracy  : {acc*100:.2f}%")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

print("\n Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# =========================================
# 7. CONFUSION MATRIX
# =========================================

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - TEST DATA")
plt.show()

In [ ]:
# MANUAL TESTING (AUTO IMAGE SELECTION - FINAL)

import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# 1. LOAD MODEL
fusion_model = load_model("best_fusion_model.keras")
print("Model loaded")

# 2. CLASS NAMES
class_names = ['Normal', 'OSCC']

# 3. USER INPUT

user_input = "OSCC"

# 4. SELECT RANDOM IMAGE FROM DATASET

folder = f"/content/dataset/test/{user_input}"

img_name = random.choice(os.listdir(folder))
img_path = os.path.join(folder, img_name)

print("Selected Image:", img_path)

# 5. LOAD & PREPROCESS IMAGE

img = tf.keras.utils.load_img(img_path, target_size=(224, 224))
img_array = tf.keras.utils.img_to_array(img)
img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)

# 6. PREDICTION

# Original prediction
pred1 = fusion_model.predict(img_array, verbose=0)

# Flipped prediction
flipped = tf.image.flip_left_right(img_array)
pred2 = fusion_model.predict(flipped, verbose=0)

# Average prediction
pred = (pred1 + pred2) / 2

normal_prob = pred[0][0]
oscc_prob = pred[0][1]

# FINAL DECISION RULE
if oscc_prob > 0.85:
    pred_class = 1   # OSCC
else:
    pred_class = 0   # Normal

confidence = 100 * max(normal_prob, oscc_prob)

# 7. DISPLAY RESULT

plt.imshow(img)
plt.axis("off")

print("\nMANUAL TEST RESULT ")
print(f"Input Class (Expected): {user_input}")
print(f"Prediction           : {class_names[pred_class]}")
# print(f"Confidence           : {confidence:.2f}%")
# print(f"Probabilities → Normal: {normal_prob:.2f}, OSCC: {oscc_prob:.2f}")

# plt.title(f"{class_names[pred_class]} ({confidence:.1f}%)")
plt.show()

In [ ]:
# =========================================
# MANUAL TESTING (UPLOAD IMAGE)
# =========================================

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from google.colab import files
from PIL import Image

# 1. LOAD MODEL
fusion_model = load_model("best_fusion_model.keras")
print("✅ Model loaded")

# 2. CLASS NAMES
class_names = ['Normal', 'OSCC']

# =========================================
# 3. UPLOAD IMAGE 🔥
# =========================================

uploaded = files.upload()

# Get uploaded file name
img_path = list(uploaded.keys())[0]
print("📂 Uploaded Image:", img_path)

# =========================================
# 4. LOAD & PREPROCESS IMAGE
# =========================================

img = Image.open(img_path).resize((224, 224))
img_array = np.array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# =========================================
# 5. PREDICTION (WITH THRESHOLD 🔥)
# =========================================

pred = fusion_model.predict(img_array, verbose=0)

normal_prob = pred[0][0]
oscc_prob = pred[0][1]

# 🔥 THRESHOLD (adjust if needed)
threshold = 0.7

if oscc_prob > threshold:
    pred_class = 1
else:
    pred_class = 0

confidence = 100 * max(normal_prob, oscc_prob)

# =========================================
# 6. DISPLAY RESULT
# =========================================

plt.imshow(img)
plt.axis("off")

print("\n🔍 MANUAL TEST RESULT")
print(f"Prediction           : {class_names[pred_class]}")
print(f"Confidence           : {confidence:.2f}%")
print(f"Probabilities → Normal: {normal_prob:.2f}, OSCC: {oscc_prob:.2f}")

plt.title(f"{class_names[pred_class]} ({confidence:.1f}%)")
plt.show()

In [ ]:
# =========================================
# MANUAL TESTING WITH VALIDATIONS (FINAL)
# =========================================

import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
import cv2

# 1. LOAD MODEL
fusion_model = load_model("best_fusion_model.keras")
print("Model loaded")

# 2. CLASS NAMES
class_names = ['Normal', 'OSCC']

# 3. USER INPUT
user_input = "Normal"   # Change to "OSCC" for testing

folder = f"/content/dataset/test/{user_input}"

# =========================================
# TEST CASE 1: NO FILE / EMPTY FOLDER
# =========================================
if not os.path.exists(folder) or len(os.listdir(folder)) == 0:
    print("Error: No images found in the folder!")
    exit()

# SELECT RANDOM IMAGE
img_name = random.choice(os.listdir(folder))
img_path = os.path.join(folder, img_name)

print("\nSelected Image:", img_path)

# =========================================
# TEST CASE 2: INVALID FILE FORMAT
# =========================================
if not img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
    print("Error: Invalid file format! Please upload an image.")
    exit()

# =========================================
# TEST CASE 3: CORRUPTED IMAGE
# =========================================
try:
    img_raw = tf.keras.utils.load_img(img_path)
except:
    print("Error: Unable to load image! File may be corrupted.")
    exit()

# =========================================
# TEST CASE 4: LOW RESOLUTION CHECK
# =========================================
print("Original Image Size:", img_raw.size)

if img_raw.size[0] < 100 or img_raw.size[1] < 100:
    print("Warning: Low resolution image! Prediction may be affected.")

# =========================================
# TEST CASE 5: BLUR DETECTION
# =========================================
img_cv = cv2.imread(img_path)
gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)

blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
print("Blur Score:", blur_score)

if blur_score < 50:
    print("Warning: Image is blurry! Prediction may not be reliable.")

# =========================================
# PREPROCESS IMAGE
# =========================================
img = tf.keras.utils.load_img(img_path, target_size=(224, 224))
img_array = tf.keras.utils.img_to_array(img)
img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)

# =========================================
# PREDICTION (WITH AUGMENTATION)
# =========================================

# Original
pred1 = fusion_model.predict(img_array, verbose=0)

# Flipped
flipped = tf.image.flip_left_right(img_array)
pred2 = fusion_model.predict(flipped, verbose=0)

# Average prediction
pred = (pred1 + pred2) / 2

normal_prob = pred[0][0]
oscc_prob = pred[0][1]

# =========================================
# TEST CASE 6: LOW CONFIDENCE (UNRELATED IMAGE)
# =========================================
confidence = max(normal_prob, oscc_prob)

if confidence < 0.6:
    print("Warning: Low confidence! Image may be unrelated or unclear.")

# FINAL DECISION RULE
if oscc_prob > 0.85:
    pred_class = 1   # OSCC
else:
    pred_class = 0   # Normal

confidence_percent = confidence * 100

# =========================================
# DISPLAY RESULT
# =========================================
plt.imshow(img)
plt.axis("off")

print("\n========== MANUAL TEST RESULT ==========")
print(f"Input Class (Expected): {user_input}")
print(f"Prediction           : {class_names[pred_class]}")
print(f"Confidence           : {confidence_percent:.2f}%")
print(f"Probabilities → Normal: {normal_prob:.2f}, OSCC: {oscc_prob:.2f}")

plt.title(f"{class_names[pred_class]} ({confidence_percent:.1f}%)")
plt.show()